In [0]:
from pyspark.sql.functions import col 
from pyspark.sql import functions as F

In [0]:
%fs ls /Volumes/gizmobox/landing/operational_data/Customers/

In [0]:
customer_df = spark.read.format('json').load('/Volumes/gizmobox/landing/operational_data/Customers/')
display(customer_df)

In [0]:
customer_df_with_metadata = customer_df.select('_metadata.file_path', '*')
display(customer_df_with_metadata)

In [0]:
customer_df_with_metadata.writeTo("gizmobox.bronze.customer_sample").createOrReplace()

In [0]:
customer_df_sample = spark.table('gizmobox.bronze.sample')
display(customer_df_sample)

In [0]:
null_removed_df = customer_df_sample.filter(customer_df_sample.customer_id.isNotNull())
display(null_removed_df)

In [0]:
customer_df_dedups = null_removed_df.dropDuplicates()
display(customer_df_dedups)

In [0]:
customer_df_dedups_ordered = customer_df_dedups.orderBy(col('customer_id'))
display(customer_df_dedups_ordered)

In [0]:
latest_customer_df_max_created = (
    customer_df_dedups_ordered.groupBy('customer_id')
    .agg(F.max('created_timestamp').alias('max_created_timestamp'))
)
display(latest_customer_df_max_created)

In [0]:
df_distinct_customer = (
    customer_df_dedups_ordered.join(latest_customer_df_max_created,
                                    (latest_customer_df_max_created.customer_id == customer_df_dedups_ordered.customer_id) &
                                    (latest_customer_df_max_created.max_created_timestamp ==
                                    customer_df_dedups_ordered.created_timestamp), 
                                    'inner').select(customer_df_dedups_ordered['*'])
)
display(df_distinct_customer)

In [0]:
df_casted_customer = (
    df_distinct_customer
        .withColumn('customer_id', col('customer_id').cast('integer'))
        .withColumn('customer_name', col('customer_name').cast('string'))
        .withColumn('email', col('email').cast('string'))
        .withColumn('telephone', col('telephone').cast('string'))
        .withColumn('created_timestamp', col('created_timestamp').cast('timestamp'))
        .withColumn('date_of_birth', col('date_of_birth').cast('date'))
        .withColumn('member_since', col('member_since').cast('date'))
)
display(df_casted_customer)

In [0]:
address_df = (
    spark.read.format('csv')
    .option('header', 'True')
    .option('delimiter', '\t')
    .load('/Volumes/gizmobox/landing/operational_data/Addresses/')
)
display(address_df)

In [0]:
df = spark.createDataFrame([
    (1, 66, '2025-01-10 11:30:00', 85.75, 'Payment Error:Retailer'),  
    (2, 69, '2025-01-03 12:40:15', 120.50, 'Order Cancelled:Customer'),  
    (3, 72, '2025-01-06 14:45:30', 65.00, 'Product Returned:Customer'),  
    (4, 73, '2025-01-07 16:10:45', 210.99, 'Order Cancelled:Customer'),  
    (5, 75, '2025-01-09 18:25:00', 45.20, 'Payment Error:Retailer'),  
    (6, 80, '2025-01-10 09:35:20', 130.15, 'Order Cancelled:Customer'),  
    (7, 83, '2025-01-12 11:20:40', 150.00, 'Product Returned:Customer'),  
    (8, 85, '2025-01-14 13:15:30', 89.99, 'Order Cancelled:Customer'),  
    (9, 89, '2025-01-15 15:00:00', 78.50, 'Payment Error:Retailer'),  
    (10, 91, '2025-01-17 16:45:15', 250.75, 'Product Returned:Customer')
    ], schema = ["refund_id", "payment_id", "refund_timestamp", "refund_amount", "refund_reason"]
)
display(df)

In [0]:

df_schema_changed = (
    df.withColumn('refund_timestamp', col('refund_timestamp').cast('timestamp'))
        .withColumn('refund_amount', col('refund_amount').cast('decimal(10,2)'))
        .withColumn('refund_id', col('refund_id').cast('integer'))
        .withColumn('payment_id', col('payment_id').cast('integer'))
)

display(df_schema_changed)

In [0]:
payment_df = spark.table("gizmobox.bronze.payments")
display(payment_df)

In [0]:
payment_df_updated = (
    payment_df.select(
        'payment_id',
        'order_id',
        F.date_format('payment_timestamp', 'yyyy-MM-dd').cast('date').alias('payment_date'),
        F.date_format('payment_timestamp', 'HH:mm:ss').alias('payment_time'),
        F.when(F.col('payment_status') == 1, 'Success')
            .when(F.col('payment_status') == 2, 'Pending')
            .when(F.col('payment_status') == 3, 'Cancelled')
            .when(F.col('payment_status') == 4, 'Failed')
            .otherwise('Unknown')
            .alias("payment_status"),
        'payment_method'
    )
)
display(payment_df_updated)

In [0]:
df = spark.table('gizmobox.bronze.py_addresses')
display(df)

In [0]:
address_df = spark.table('gizmobox.bronze.py_addresses')
display(address_df)

In [0]:
address_pivoted_df = (
    address_df.groupBy("customer_id")
    .pivot('address_type', ['shipping', 'billing'])
    .agg(
        F.max('address_line_1').alias('address_line_1'),
        F.max('city').alias('city'),
        F.max('state').alias('state'),
        F.max('postcode').alias('postcode')
    )
    .orderBy('customer_id')
)
display(address_pivoted_df)

In [0]:
silver_orders = spark.table("gizmobox.silver.orders")
silver_payments = spark.table("gizmobox.silver.payments")
silver_refunds = spark.table("hive_metastore.silver.refunds")

In [0]:
silver_orders = (
    silver_orders
    .withColumn("order_month", F.date_format('transaction_timestamp', 'yyyy-MM'))
    .groupBy('order_month', 'customer_id')
    .agg(
        F.countDistinct('order_id').alias('total_orders'),
        F.sum('quantity').alias('total_items_bought'),
        F.sum(F.col('price') * F.col('quantity')).alias('total_amount')
    )
)
display(silver_orders)

In [0]:
customer_df = customer_df_dedups_ordered
customer_max_df = latest_customer_df_max_created
display(customer_df)
display(customer_max_df)